# Day 09. Exercise 00
# Regularization

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
import sklearn
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from IPython.display import HTML, display

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.

In [144]:
df = pd.read_csv("../data/dayofweek.csv")
df.head(3)

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [145]:
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [146]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.

In [147]:
model_logreg = LogisticRegression(fit_intercept=False, random_state=21, solver='saga')

2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model

The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [179]:
%%time

skf = StratifiedKFold(n_splits=10)  # убрала shuffle=True, random_state=21 !!

cv_results = cross_validate(
    model_logreg,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc = cv_results["test_score"].mean()
std_acc = cv_results["test_score"].std()

for train_score, valid_score in zip(
    cv_results["train_score"], cv_results["test_score"]
):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base Logistic Regression on crossval is {mean_acc:.5f}
Std is {std_acc:.5f}\033[0m""")

train -   0.62902   |   valid -   0.59259
train -   0.64633   |   valid -   0.62963
train -   0.63479   |   valid -   0.56296
train -   0.65622   |   valid -   0.61481
train -   0.63397   |   valid -   0.57778
train -   0.64056   |   valid -   0.59259
train -   0.64221   |   valid -   0.65926
train -   0.65952   |   valid -   0.56296
train -   0.64333   |   valid -   0.59701
train -   0.63674   |   valid -   0.62687
Average accuracy of Base Logistic Regression on crossval is 0.60165
Std is 0.02943
CPU times: user 28.6 ms, sys: 25.8 ms, total: 54.4 ms
Wall time: 198 ms


In [149]:
# cv_results

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [150]:
%%time

# из документации:
# solvers = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
# penalty=None: все, кроме liblinear)
# penalty='l1' (штраф использует модули весов (|w|) и зануляет коэффициенты у неважных или шумовых признаков, выполняя автоматический отбор фичей.): работает только с liblinear и saga
# penalty='l2' (штраф использует квадраты весов (w²), плавно уменьшает веса признаков ближе к нулю, но никогда не зануляет их полностью): все солверы

penalty_combos = {
    None: ["lbfgs", "newton-cg", "newton-cholesky", "sag", "saga"],
    "l1": ["saga"],  # убрала из списка"liblinear" после получения ошибки см.ниже
    "l2": [
        "lbfgs",
        "newton-cg",
        "newton-cholesky",
        "sag",
        "saga",
    ],  # убрала из списка"liblinear" после получения ошибки см.ниже
}
# ValueError: The 'liblinear' solver does not support multiclass classification (n_classes >= 3). Either use another solver or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.

# согласно новым правилам библиотеки sklearn вместо penalty='l2' нужно использовать l1_ratio=0 (а параметр penalty оставить по умолчанию).
# Вместо penalty='l1' нужно использовать l1_ratio=1.
# Вместо penalty=None нужно использовать C=np.inf (бесконечный коэффициент, убирающий штраф).
# если забить, то куча warnings мешает читать вывод принта.
# либо есть вариант юзать import warnings
# warnings.filterwarnings("ignore", category=FutureWarning)

import warnings

warnings.filterwarnings("ignore")

results = []

for key in penalty_combos:
    for val in penalty_combos[key]:
        model_logreg = LogisticRegression(
            penalty=key, solver=val, fit_intercept=False, random_state=21, max_iter=2000
        )

        cv_res_regs = cross_validate(
            model_logreg, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
        )

        mean_acc = cv_res_regs["test_score"].mean()
        std_acc = cv_res_regs["test_score"].std()

        results.append(
            {
                "Penalty": "None" if key is None else key,
                "Solver": val,
                "Mean CV Accuracy": round(mean_acc, 5),
                "Std": round(std_acc, 5),
            }
        )
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Penalty          Solver  Mean CV Accuracy     Std
   None           lbfgs           0.62388 0.03392
   None       newton-cg           0.62462 0.03379
   None newton-cholesky           0.62388 0.03392
   None             sag           0.62462 0.03379
   None            saga           0.62462 0.03379
     l1            saga           0.59646 0.02848
     l2           lbfgs           0.60239 0.02852
     l2       newton-cg           0.60091 0.03047
     l2 newton-cholesky           0.60165 0.02943
     l2             sag           0.60165 0.02943
     l2            saga           0.60165 0.02943
CPU times: user 318 ms, sys: 65.1 ms, total: 383 ms
Wall time: 5.73 s


In [151]:
%%time

# обновленный вариант под современную библу
common_solvers = ["lbfgs", "newton-cg", "newton-cholesky", "sag", "saga"]
configs = {None: common_solvers, "l2": common_solvers, "l1": ["saga"]}
results = []

for penalty, solvers in configs.items():
    for solver in solvers:
        model = LogisticRegression(
            penalty=penalty,
            solver=solver,
            fit_intercept=False,
            random_state=21,
            max_iter=2000,
        )

        cv_res = cross_validate(
            model, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
        )

        results.append(
            {
                "Penalty": "None" if penalty is None else penalty,
                "Solver": solver,
                "Mean CV Accuracy": round(cv_res["test_score"].mean(), 5),
                "Std": round(cv_res["test_score"].std(), 5),
            }
        )
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))  # либо print(df_results.set_index("Penalty"))

Penalty          Solver  Mean CV Accuracy     Std
   None           lbfgs           0.62388 0.03392
   None       newton-cg           0.62462 0.03379
   None newton-cholesky           0.62388 0.03392
   None             sag           0.62462 0.03379
   None            saga           0.62462 0.03379
     l2           lbfgs           0.60239 0.02852
     l2       newton-cg           0.60091 0.03047
     l2 newton-cholesky           0.60165 0.02943
     l2             sag           0.60165 0.02943
     l2            saga           0.60165 0.02943
     l1            saga           0.59646 0.02848
CPU times: user 255 ms, sys: 54.8 ms, total: 310 ms
Wall time: 5.1 s


In [152]:
# requirements по ТЗ: scikit-learn (0.23.1)
# солвер по умолчанию там был lbfgs. в версиях 0.21 и до  - liblinear

# Мультикласс по умолчанию (multi_class): В версии 0.23.1 параметр multi_class имел значение по умолчанию 'auto'. Для мультиклассовых задач это означало, что если выбран солвер lbfgs, библиотека автоматически включала режим 'multinomial' (многочленная логистическая регрессия, которая минимизирует полный multinomial loss).

# Почему цифры в переборе (0.60165) совпали на солверах sag, saga и newton-cholesky, но не совпали на lbfgs?Математический алгоритм lbfgs в современной версии 1.9.0 был сильно оптимизирован на уровне C-кода по сравнению с древней версией 0.23.1. Из-за этого на тех же самых данных «современный» lbfgs считает шаги чуть точнее и выдает 0.60239. А вот алгоритмы sag и saga за эти годы в scikit-learn математически не менялись — именно поэтому они выдали вам абсолютно идентичный «привет из прошлого» в виде 0.60165

print(f'установленная у меня версия библиотеки sklearn: {sklearn.__version__}')

установленная у меня версия библиотеки sklearn: 1.9.0


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [153]:
model_svc_default = SVC(kernel="linear", random_state=21, probability=True)

In [180]:
%%time

skf_svc = StratifiedKFold(n_splits=10)

cv_res_svc = cross_validate(
    model_svc_default,
    X_train,
    y_train,
    cv=skf_svc,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc_svc = cv_res_svc["test_score"].mean()
std_acc_svc = cv_res_svc["test_score"].std()

for train_score, valid_score in zip(
    cv_res_svc["train_score"], cv_res_svc["test_score"]
):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base SVM Model on crossval is {mean_acc_svc:.5f}
Std is {std_acc_svc:.5f}\033[0m""")

train -   0.70486   |   valid -   0.65926
train -   0.69662   |   valid -   0.75556
train -   0.69415   |   valid -   0.62222
train -   0.70239   |   valid -   0.65185
train -   0.69085   |   valid -   0.65185
train -   0.68920   |   valid -   0.64444
train -   0.69250   |   valid -   0.72593
train -   0.70074   |   valid -   0.62222
train -   0.69605   |   valid -   0.61940
train -   0.71087   |   valid -   0.63433
Average accuracy of Base SVM Model on crossval is 0.65871
Std is 0.04359
CPU times: user 43.9 ms, sys: 10.7 ms, total: 54.7 ms
Wall time: 451 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [158]:
%%time

c = [0.01, 0.1, 1, 10, 50, 100, 500]
results_svc = []

for reg_parameter in c:
    model_svc_optimized = SVC(kernel="linear", random_state=21, probability=True, C=reg_parameter)

    cv_res_svc = cross_validate(
    model_svc_optimized,
    X_train,
    y_train,
    cv=skf_svc,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)
    mean_acc = cv_res_svc["test_score"].mean()
    std_acc = cv_res_svc["test_score"].std()

    results_svc.append(
        {
            "C": reg_parameter,
            "Mean accuracy": round(mean_acc, 5),
            "Std": round(std_acc, 5),
        }
    )
df_res_svc = pd.DataFrame(results_svc)
print(df_res_svc.to_string(index=False)) 

     C  Mean accuracy     Std
  0.01        0.37534 0.01848
  0.10        0.56230 0.02177
  1.00        0.65871 0.04359
 10.00        0.72771 0.04417
 50.00        0.75441 0.03438
100.00        0.75589 0.03550
500.00        0.75441 0.03357
CPU times: user 253 ms, sys: 100 ms, total: 353 ms
Wall time: 17.4 s


ВЫВОД: Модель достигает своего максимума при C = 100.00 с точностью 0.75589. При дальнейшем увеличении C до 500.00 точность перестает расти и даже немного откатывается назад (до 0.75441). Это сигнал того, что модель начинает избыточно подстраиваться под обучающую выборку. Стандартное отклонение заметно снизилось по сравнению с базовой моделью (упало с 0.04359 до 0.035). Это значит, что модель стала не просто точнее, но и стабильнее ведет себя на разных фолдах.

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [163]:
model_dt = DecisionTreeClassifier(max_depth=10, random_state=21)

In [182]:
%%time

cv_res_dt = cross_validate(
    model_dt,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc_dt = cv_res_dt["test_score"].mean()
std_acc_dt = cv_res_dt["test_score"].std()

for train_score, valid_score in zip(
    cv_res_dt["train_score"], cv_res_dt["test_score"]
):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base Decision Tree Model on crossval is {mean_acc_dt:.5f},
Std is {std_acc_dt:.5f}\033[0m""")
# \033[32m — включает зеленый цвет, \033[0m — сбрасывает цвет обратно на стандартный

train -   0.81039   |   valid -   0.74074
train -   0.77741   |   valid -   0.74074
train -   0.83347   |   valid -   0.70370
train -   0.79720   |   valid -   0.76296
train -   0.82440   |   valid -   0.75556
train -   0.80379   |   valid -   0.68889
train -   0.80709   |   valid -   0.76296
train -   0.80132   |   valid -   0.65926
train -   0.80807   |   valid -   0.75373
train -   0.80478   |   valid -   0.68657
Average accuracy of Base Decision Tree Model on crossval is 0.72551,
Std is 0.03562
CPU times: user 47.2 ms, sys: 8.7 ms, total: 55.9 ms
Wall time: 81.7 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.